# 02. Exploratory Data Analysis (EDA)

This notebook reads the synthetic medication dataset, validates the schema, computes adherence metrics, and produces simple visual summaries. The dataset is intentionally synthetic and must not be treated as real medical data.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ml.src.preprocessing import create_processed_dataset, create_ml_ready_dataset

processed_path = repo_root / "ml" / "data" / "processed" / "processed_medication_data.csv"
ml_ready_path = repo_root / "ml" / "data" / "processed" / "ml_ready_medication_data.csv"

if not processed_path.exists():
    create_processed_dataset(output_path=processed_path)
if not ml_ready_path.exists():
    create_ml_ready_dataset(output_path=ml_ready_path)

df = pd.read_csv(processed_path)
print(f"Loaded processed dataset from: {processed_path}")
print(f"Loaded rows: {len(df)}")
print(df.head())

In [ ]:
# Basic dataset overview
num_rows = len(df)
num_patients = df["patient_id"].nunique()
num_medicines = df["medicine_id"].nunique()
start_date = df["scheduled_date"].min()
end_date = df["scheduled_date"].max()
missing_values = df.isna().sum()
duplicate_rows = df.duplicated().sum()
unique_statuses = sorted(df["dose_status"].dropna().unique().tolist())

summary = {
    "rows": num_rows,
    "patients": num_patients,
    "medicines": num_medicines,
    "date_range_start": start_date,
    "date_range_end": end_date,
    "missing_values": missing_values.to_dict(),
    "duplicate_records": duplicate_rows,
    "unique_dose_statuses": unique_statuses,
}

print(summary)

In [ ]:
# Adherence metrics
# Adherence % = (Taken doses / Total scheduled doses) * 100

total_scheduled = len(df)
total_taken = (df["dose_status"] == "TAKEN").sum()
total_missed = (df["dose_status"] == "MISSED").sum()
total_pending = (df["dose_status"] == "PENDING").sum()

adherence_pct = (total_taken / total_scheduled * 100) if total_scheduled else 0.0

adherence_summary = {
    "total_scheduled_doses": total_scheduled,
    "total_taken_doses": total_taken,
    "total_missed_doses": total_missed,
    "total_pending_doses": total_pending,
    "overall_adherence_percentage": round(adherence_pct, 2),
}
print(adherence_summary)

In [ ]:
# Delay analysis

# Delay values are expected in minutes and should be non-negative.
delay_series = pd.to_numeric(df["delay_minutes"], errors="coerce").dropna()

if len(delay_series) > 0:
    delay_summary = {
        "average_delay_minutes": round(float(delay_series.mean()), 2),
        "median_delay_minutes": round(float(delay_series.median()), 2),
        "max_delay_minutes": int(delay_series.max()),
        "min_delay_minutes": int(delay_series.min()),
    }
    print(delay_summary)
    print(delay_series.describe())
else:
    print("No valid delay values available for analysis.")

In [ ]:
# Time-based adherence analysis

df["day_of_week"] = df["day_of_week"].astype(str)
df["time_period"] = df["time_period"].astype(str)

period_summary = (
    df.groupby("time_period")["dose_status"]
      .value_counts()
      .unstack(fill_value=0)
      .reindex(["MORNING", "AFTERNOON", "EVENING", "NIGHT"], fill_value=0)
)
print(period_summary)

day_summary = (
    df.groupby("day_of_week")["dose_status"]
      .value_counts()
      .unstack(fill_value=0)
      .reindex(["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY", "SUNDAY"], fill_value=0)
)
print(day_summary)

period_pct = (
    df.groupby("time_period")["dose_status"]
      .apply(lambda s: (s.eq("TAKEN").sum() / len(s)) * 100)
      .reindex(["MORNING", "AFTERNOON", "EVENING", "NIGHT"], fill_value=0)
)

weekday_pct = (
    df.groupby("day_of_week")["dose_status"]
      .apply(lambda s: (s.eq("TAKEN").sum() / len(s)) * 100)
      .reindex(["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY", "SUNDAY"], fill_value=0)
)

In [ ]:
# Generate simple charts

status_counts = df["dose_status"].value_counts().reindex(["TAKEN", "PENDING", "MISSED"], fill_value=0)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Taken vs missed doses
status_counts.plot(kind="bar", ax=axes[0, 0], color=["#2ca02c", "#ff7f0e", "#d62728"])
axes[0, 0].set_title("Taken vs Missed vs Pending Doses")
axes[0, 0].set_xlabel("Dose status")
axes[0, 0].set_ylabel("Count")

# 2. Overall adherence percentage
adherence_bar = axes[0, 1]
adherence_bar.bar(["Overall adherence"], [adherence_pct], color="#4c72b0")
adherence_bar.set_ylim(0, 100)
adherence_bar.set_title("Overall Adherence Percentage")
adherence_bar.set_ylabel("%")

# 3. Adherence by time period
period_pct.plot(kind="bar", ax=axes[0, 2], color="#6baed6")
axes[0, 2].set_title("Adherence by Time Period")
axes[0, 2].set_ylabel("Adherence %")

# 4. Adherence by day of week
weekday_pct.plot(kind="bar", ax=axes[1, 0], color="#9ecae1")
axes[1, 0].set_title("Adherence by Day of Week")
axes[1, 0].set_ylabel("Adherence %")

# 5. Distribution of delay minutes
pd.to_numeric(df["delay_minutes"], errors="coerce").plot.hist(ax=axes[1, 1], bins=10, color="#bcbd22")
axes[1, 1].set_title("Distribution of Delay Minutes")
axes[1, 1].set_xlabel("Delay minutes")
axes[1, 1].set_ylabel("Frequency")

# 6. Patient-level adherence
patient_adherence = (
    df.groupby("patient_id").apply(lambda g: (g["dose_status"].eq("TAKEN").sum() / len(g)) * 100)
      .sort_values()
)
patient_adherence.plot(kind="bar", ax=axes[1, 2], color="#8c564b")
axes[1, 2].set_title("Patient-Level Adherence")
axes[1, 2].set_ylabel("Adherence %")

plt.tight_layout()
plt.savefig(repo_root / "ml" / "reports" / "eda_charts.png", dpi=200)
print("Saved EDA chart image to ml/reports/eda_charts.png")

## Notes

This notebook is a working foundation for adherence analysis and is intentionally limited to clean, reproducible exploration. It does not make clinical claims or use fake medical conclusions.